# Statistical Time Series Forecasting — Notebook 10
## End-to-End Forecasting Workflow

**Goal:** Tie everything together. This notebook is a *template* you'll reuse on every forecasting project.

---

### The recommended workflow

```
1. EDA              ─→ Plot, decompose, infer frequency & seasonality
2. Stationarise     ─→ ADF/KPSS, transform & difference as needed
3. Identify         ─→ ACF/PACF → candidate orders
4. Split            ─→ Chronological train/test split
5. Baselines        ─→ Naive, seasonal naive, mean, drift
6. Models           ─→ ETS family, ARIMA, SARIMA, SARIMAX
7. Diagnostics      ─→ Residuals, Ljung-Box, AIC/BIC
8. Compare          ─→ Metrics table, MASE vs naive
9. Forecast         ─→ Refit on full data, project forward with CI
```


## Setup
Run this once per Colab session. Make sure `tsf_utils.py` is uploaded to your runtime, and your dataset is accessible.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

# pio.renderers.default = "colab"   # uncomment if plots don't render in Colab
pio.templates.default = "plotly_white"

import tsf_utils as tsf


## Step 1 — Load & inspect

In [ ]:
DATA_PATH = "your_dataset.csv"
DATE_COL  = "Date"
VALUE_COL = "Sales"
FREQ      = "MS"
HORIZON   = 12          # how far into the future to forecast

series = tsf.load_timeseries(DATA_PATH, DATE_COL, VALUE_COL, FREQ,
                             fill_method="interpolate")

S = tsf.infer_seasonal_period(series)
print(f"N={len(series)}, freq={series.index.freqstr}, seasonal period={S}")
tsf.plot_timeseries(series, title=VALUE_COL).show()

## Step 2 — Decomposition

In [ ]:
if S > 1:
    _, fig = tsf.decompose_and_plot(series, method="stl")
    fig.show()

## Step 3 — Stationarity check

In [ ]:
tsf.stationarity_report(series)

## Step 4 — Train/test split

In [ ]:
train, test = tsf.train_test_split_ts(series, test_size=0.2)
h = len(test)
print(f"Train={len(train)}  Test={h}")

## Step 5 — Baselines

In [ ]:
forecasts = {
    "Naive":  tsf.naive_forecast(train, h),
    "Mean":   tsf.mean_forecast(train, h),
    "Drift":  tsf.drift_forecast(train, h),
}
if S > 1:
    forecasts["SeasonalNaive"] = tsf.seasonal_naive_forecast(train, h, S)
for k in forecasts:
    forecasts[k].index = test.index

## Step 6 — Model zoo: ETS, ARIMA, SARIMA

In [ ]:
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

ets_kwargs = dict(error="add", trend="add", damped_trend=True)
if S > 1: ets_kwargs.update(seasonal="add", seasonal_periods=S)
ets_fit = ETSModel(train, **ets_kwargs).fit(disp=False)
ets_fc  = ets_fit.forecast(h); ets_fc.index = test.index
forecasts["ETS"] = ets_fc

try:
    from pmdarima import auto_arima
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pmdarima", "-q"])
    from pmdarima import auto_arima

arima_auto = auto_arima(train, seasonal=False, stepwise=True,
                        suppress_warnings=True, error_action="ignore")
forecasts["ARIMA"] = pd.Series(arima_auto.predict(n_periods=h), index=test.index)

if S > 1:
    sarima_auto = auto_arima(train, seasonal=True, m=S, stepwise=True,
                             suppress_warnings=True, error_action="ignore")
    forecasts["SARIMA"] = pd.Series(sarima_auto.predict(n_periods=h), index=test.index)

print("Models fitted:", list(forecasts.keys()))

## Step 7 — Comparison table

In [ ]:
cmp = tsf.compare_models(train, test, forecasts, seasonality=max(S, 1)).sort_values("MAE")
cmp

## Step 8 — Visualise all models on one chart

In [ ]:
tsf.plot_multi_forecast(train, test, forecasts,
                        title="All models vs actuals").show()

## Step 9 — Diagnose the chosen model

In [ ]:
best_name = cmp.index[0]
best_fc   = forecasts[best_name]
print(f"Best by MAE: {best_name}")

resid = test - best_fc
tsf.plot_residual_diagnostics(resid).show()
_ = tsf.ljung_box_test(resid, lags=10)

## Step 10 — Refit on full data, forecast the future

In [ ]:
if best_name == "ETS":
    final = ETSModel(series, **ets_kwargs).fit(disp=False)
    fc_full = final.forecast(HORIZON)
elif best_name == "ARIMA":
    final = auto_arima(series, seasonal=False, stepwise=True,
                       suppress_warnings=True, error_action="ignore")
    fc_full = pd.Series(final.predict(n_periods=HORIZON))
elif best_name == "SARIMA":
    final = auto_arima(series, seasonal=True, m=S, stepwise=True,
                       suppress_warnings=True, error_action="ignore")
    fc_full = pd.Series(final.predict(n_periods=HORIZON))
else:
    if best_name == "SeasonalNaive":
        fc_full = tsf.seasonal_naive_forecast(series, HORIZON, S)
    elif best_name == "Drift":
        fc_full = tsf.drift_forecast(series, HORIZON)
    elif best_name == "Mean":
        fc_full = tsf.mean_forecast(series, HORIZON)
    else:
        fc_full = tsf.naive_forecast(series, HORIZON)

last = series.index[-1]
freq = series.index.freq or pd.infer_freq(series.index)
future_idx = pd.date_range(start=last, periods=HORIZON+1, freq=freq)[1:]
fc_full.index = future_idx

fig = go.Figure()
fig.add_trace(go.Scatter(x=series.index, y=series.values, mode="lines",
                         name="History", line=dict(color="#3498db", width=1.4)))
fig.add_trace(go.Scatter(x=fc_full.index, y=fc_full.values, mode="lines",
                         name=f"{best_name} forecast (next {HORIZON})",
                         line=dict(color="#e74c3c", width=2, dash="dash")))
fig.update_layout(title=f"Final forecast — {best_name}",
                  xaxis_title="Date", yaxis_title=VALUE_COL,
                  height=460, hovermode="x unified")
fig.show()

print("\nForecast values:")
print(fc_full.round(2))

## Common pitfalls — read before shipping a forecast

1. **Used a random train/test split** — the model has seen the future.
2. **Reported only MAPE on a near-zero series** — MAPE explodes; use sMAPE or MASE.
3. **Ignored residual autocorrelation** — your prediction intervals are too tight.
4. **Forecasted multiplicative seasonality with an additive model** — under/over-shoots at peaks.
5. **Forgot to re-fit on the full data** — final forecast was generated by a stale model.
6. **No baseline comparison** — can't claim the model is good without showing it beats `seasonal_naive`.
7. **Treated SARIMAX exogenous forecasts as free** — you need future regressors, themselves forecasts.
8. **Overfit by tuning on the test set** — keep a third `validation` slice if you tune extensively.
9. **Ignored regime shifts** — pandemic, structural break, new regulation. Sometimes you need to retrain on recent data only.

## Where to go next

- **Forecasting Principles & Practice** (Hyndman & Athanasopoulos) — free book: <https://otexts.com/fpp3/>
- **Prophet** (Facebook) and **NeuralProphet** — additive-decomposition models with built-in calendar/holiday effects.
- **Tree-based regressors with engineered lag features** — XGBoost/LightGBM. Often beats classical models on rich datasets.
- **Deep learning** — DeepAR, N-BEATS, TFT. Worth the effort with many series or rich exogenous structure.
